# <center><u><b>L'analyse de performance e-commerce — TheLook Europe</b></u></center>


# <b>Contexte</b>

Vous assumez la position d'Analyste de données chez « TheLook Europe ». L'équipe du commerce électronique vous sollicite pour examiner la performance de l'activité dans un cadre spécifique et pour établir une comparaison entre l'année 2023 et l'année 2024. Votre tâche consiste à analyser les dynamiques de chiffre d'affaires, de marge, de retours et de comportement du client, puis à communiquer des conclusions claires à travers un tableau de bord Power BI.


# <b>L'objectif</b>

L'objectif de cette analyse est d'étudier les performances e-commerce du département Women en France entre 2023 et 2024.

Nous allons :

- contrôler la qualité des données,
- réaliser une analyse exploratoire,
- calculer les KPI business,
- comparer les performances 2023 vs 2024,
- identifier des recommandations business.


# <b>Le périmètre  d'étude</b>

| Paramètre | Valeur |
|---|---|
| Pays | France |
| Département | Women |
| Période | 01/01/2023 → 31/12/2024 |
| Source | `thelook_fr_women_2023_2024.csv` |
| Statut vente | `item_status = 'Complete'` |
| Statut retour | `item_status = 'Returned'` |


# <b>Les KPI à calculer</b>

| KPI | Définition |
|---|---|
| Chiffre d'affaires | `SUM(sale_price)` sur les lignes Complete |
| Marge brute | `SUM(sale_price - cost)` sur les lignes Complete |
| Panier moyen | CA / nombre de commandes distinctes (Complete) |
| Taux de retour | Returned / (Complete + Returned) × 100 |
| Taux de ré-achat | Clients avec ≥ 2 commandes Complete sur une même année / total clients actifs × 100 |


# <b>La source</b>

Fichier CSV extrait : `thelook_fr_women_2023_2024`

# 1. Configuration de l'environnement

### 1.1. Import des librairies

In [20]:
# Import des librairies avec leurs alias standard
import pandas as pd                 # Manipulation des données
import numpy as np                  # Calculs Numériques
import matplotlib.pyplot as plt     # Création des graphiques
import seaborn as sns               # Graphiques statistiques avancés

# Vérification des versions
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

# Configuration de l'affichage pandas
# display.max_columns=None   : affiche toutes les colonnes sans troncatu
# display.max_rows=100       : limite l'affichage à 100 lignes maximum
# display.precision=2        : limite l'affichage des décimales à 2 chiffres
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

Pandas version: 2.3.3
NumPy version: 2.4.4
Seaborn version: 0.13.2
Matplotlib version: 3.10.8


# 2. Importation du Dataset

### 2.1 Chargement du Dataset

In [21]:
# Import du dataset
df = pd.read_csv('/Users/ko/Documents/thelook-da-projet-soutenance/thelook-da-projet/data/thelook_fr_women_2023_2024.csv', encoding="utf-8")

# Afficher les dimensions du dataset
print(f"Dimensions du dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes")

# Afficher 5 lignes aléatoires du dataset
df.sample(5)


Dimensions du dataset : 1679 lignes x 20 colonnes


,order_id,order_item_id,product_id,item_created_at,item_status,sale_price,cost,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,user_id,gender,country,state,city
412,5207,7449,9186,2023-08-19 04:31:03+00:00,Returned,12.99,5.30,Socks & Hosiery,Women,K. Bell,K. Bell Socks Women's 2 Pack Animal Print Gift...,Returned,2023-08-16 05:49:00+00:00,2023-08-17 18:26:00+00:00,2023-08-19 07:30:00+00:00,4289,F,France,Île-de-France,Meudon
1585,33115,48051,2119,2024-12-01 02:37:20+00:00,Complete,99.99,44.80,Fashion Hoodies & Sweatshirts,Women,Ralph Lauren,RLX by Ralph Lauren Women Fashion Cashmere Hoo...,Complete,2024-11-28 04:12:00+00:00,2024-11-29 05:37:00+00:00,2024-12-01 22:13:00+00:00,26587,F,France,Auvergne-Rhône-Alpes,La Roche-sur-Foron
938,19362,28019,10898,2024-04-21 01:26:55+00:00,Processing,15.00,7.33,Intimates,Women,Wacoal,Wacoal Women's B-Smooth Hi Cut Brief,Processing,2024-04-21 03:51:00+00:00,NaN,NaN,15596,F,France,Bretagne,Vitré
51,33123,48067,2502,2023-02-01 16:42:22+00:00,Complete,12.99,5.90,Active,Women,Fox River Socks,Fox River Military Wick Dry Maximum Mid Calf B...,Complete,2023-02-01 16:46:00+00:00,2023-02-02 05:45:00+00:00,2023-02-05 00:28:00+00:00,26592,F,France,Bretagne,Quimper
1539,3714,5304,15040,2024-11-21 00:21:54+00:00,Cancelled,39.98,16.99,Maternity,Women,Motherhood Maternity,Motherhood Maternity: Sleeveless Belted Matern...,Cancelled,2024-11-19 00:56:00+00:00,NaN,NaN,3055,F,France,Île-de-France,Coupvray


### 2.2 Aperçu de la structure du dataset

In [22]:
# Affichage des informations complètes du Dataset
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          1679 non-null   int64  
 1   order_item_id     1679 non-null   int64  
 2   product_id        1679 non-null   int64  
 3   item_created_at   1679 non-null   object 
 4   item_status       1679 non-null   object 
 5   sale_price        1679 non-null   float64
 6   cost              1679 non-null   float64
 7   category          1679 non-null   object 
 8   department        1679 non-null   object 
 9   brand             1677 non-null   object 
 10  product_name      1679 non-null   object 
 11  order_status      1679 non-null   object 
 12  order_created_at  1679 non-null   object 
 13  shipped_at        1133 non-null   object 
 14  delivered_at      636 non-null    object 
 15  user_id           1679 non-null   int64  
 16  gender            1679 non-null   object 


In [23]:
# Statistiques descriptives des colonnes numériques
df.describe(include='all')

,order_id,order_item_id,product_id,item_created_at,item_status,sale_price,cost,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,user_id,gender,country,state,city
count,1679.00,1679.00,1679.00,1679,1679,1679.00,1679.00,1679,1679,1677,1679,1679,1679,1133,636,1679.00,1679,1679,1679,1679
unique,NaN,NaN,NaN,1679,5,NaN,NaN,22,1,657,1559,5,1117,747,416,NaN,1,1,13,547
top,NaN,NaN,NaN,2023-01-01 06:18:03+00:00,Shipped,NaN,NaN,Intimates,Women,Allegra K,Shadowline Short Robe,Shipped,2023-10-09 09:00:00+00:00,2024-02-04 14:10:00+00:00,2023-10-03 22:56:00+00:00,NaN,F,France,Île-de-France,Paris
freq,NaN,NaN,NaN,1,497,NaN,NaN,247,1679,101,3,497,4,4,4,NaN,1679,1679,397,94
mean,60851.68,88372.70,7922.51,NaN,NaN,57.02,27.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48702.84,NaN,NaN,NaN,NaN
std,35655.50,51820.58,4680.55,NaN,NaN,69.68,31.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28441.85,NaN,NaN,NaN,NaN
min,359.00,517.00,12.00,NaN,NaN,1.82,0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,279.00,NaN,NaN,NaN,NaN
25%,30357.00,44069.50,3812.00,NaN,NaN,19.99,9.68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24349.00,NaN,NaN,NaN,NaN
50%,60827.00,88353.00,7907.00,NaN,NaN,38.00,18.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48715.00,NaN,NaN,NaN,NaN
75%,90911.00,132106.00,12053.00,NaN,NaN,68.00,33.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72541.00,NaN,NaN,NaN,NaN


<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Analyse du describe :

#### Analyse de la catégorie "Intimates"
La catégorie *Intimates* apparaît comme la catégorie la plus représentée du dataset. Une hypothèse métier plausible est que ce type de produit bénéficie d'une fréquence d'achat plus élevée que d'autres catégories textile, notamment en raison de besoins de renouvellement réguliers.

#### Analyse des prix de vente
Le prix de vente moyen des produits est d'environ 57 €, tandis que la médiane est de 38 €. L'écart important entre la moyenne et la médiane suggère une distribution asymétrique à droite, probablement influencée par quelques produits à forte valeur (max observé : 903 €).

#### Analyse des coûts produits
Le coût moyen (~27 €) reste inférieur au prix moyen de vente (~57 €). Cette structure est cohérente avec un modèle économique e-commerce reposant sur une marge commerciale positive.

#### Analyse des statuts de commande
Le dataset contient 5 statuts différents. Il sera important de bien définir les statuts utilisés dans les KPI : **"Complete"** pour les ventes finalisées, **"Returned"** pour les retours produits.

#### Analyse des valeurs manquantes
Les colonnes `shipped_at` et `delivered_at` présentent un volume important de valeurs manquantes. Ces absences sont cohérentes avec le cycle opérationnel des commandes (annulées, retournées, en cours).
</details>

### 2.3 Dictionnaire des données

Le dictionnaire de données permet de comprendre rapidement le rôle de chaque colonne, vérifier les types de données et faciliter la lecture du projet. C'est une bonne pratique pour documenter et sécuriser l'analyse.

| Colonne | Type | Description | Exemple |
|---|---|---|---|
| order_id | int64 | Identifiant de commande | 19425 |
| order_item_id | int64 | Identifiant unique de ligne produit | 28112 |
| product_id | int64 | Identifiant produit | 6983 |
| item_created_at | datetime | Date de création de la ligne produit | 2023-01-01 06:18:03 |
| item_status | object | Statut de la ligne produit | Complete |
| sale_price | float64 | Prix de vente TTC du produit | 29.50 |
| cost | float64 | Coût du produit | 16.05 |
| category | object | Catégorie produit | Shorts |
| department | object | Département produit | Women |
| brand | object | Marque produit | Fox |
| product_name | object | Nom du produit | Fox Juniors Momentum Short |
| order_status | object | Statut global de la commande | Shipped |
| order_created_at | datetime | Date de création de la commande | 2023-01-03 |
| shipped_at | datetime | Date d'expédition | 2023-01-05 |
| delivered_at | datetime | Date de livraison | 2023-01-09 |
| user_id | int64 | Identifiant client | 15644 |
| gender | object | Genre du client | F |
| country | object | Pays du client | France |
| state | object | Région du client | Île-de-France |
| city | object | Ville du client | Paris |

# 3. Contrôle de qualité des données

### 3.1 Création d'une copie de travail du dataset

In [24]:
'''
Création d'une copie du dataset original.
Cela permet de conserver les données brutes intactes en cas d'erreur
pendant le nettoyage ou les transformations.
On travaillera uniquement sur cette copie pour sécuriser l'analyse.
'''

# Création de la copie du dataset
df_clean = df.copy()

print("Copie créée avec succès.")
print(f"Dimensions : {df_clean.shape[0]} lignes x {df_clean.shape[1]} colonnes")

Copie créée avec succès.
Dimensions : 1679 lignes x 20 colonnes


### 3.2 Traitement des valeurs manquantes

In [25]:
# Nombre de valeurs manquantes par colonne
nbr_nan = df_clean.isna().sum()

# Affichage
nbr_nan

order_id               0
order_item_id          0
product_id             0
item_created_at        0
item_status            0
sale_price             0
cost                   0
category               0
department             0
brand                  2
product_name           0
order_status           0
order_created_at       0
shipped_at           546
delivered_at        1043
user_id                0
gender                 0
country                0
state                  0
city                   0
dtype: int64

In [26]:
# Pourcentage de valeurs manquantes par colonne
pourcent_nan = (df_clean.isna().sum() / len(df_clean)) * 100 

# Affichage 
pourcent_nan

order_id             0.00
order_item_id        0.00
product_id           0.00
item_created_at      0.00
item_status          0.00
sale_price           0.00
cost                 0.00
category             0.00
department           0.00
brand                0.12
product_name         0.00
order_status         0.00
order_created_at     0.00
shipped_at          32.52
delivered_at        62.12
user_id              0.00
gender               0.00
country              0.00
state                0.00
city                 0.00
dtype: float64

<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observations :

- Les colonnes `shipped_at` et `delivered_at` présentent un nombre important de valeurs manquantes. Cela peut signifier que la commande a été annulée, retournée ou n'a pas encore été expédiée.

- Ces valeurs manquantes sont cohérentes avec le cycle de vie d'une commande e-commerce. On ne les supprime pas.

- Les valeurs manquantes dans la colonne `brand` seront remplacées par "Unknown" pour conserver les lignes dans les analyses.
</details>

#### 3.2.1 Imputation des valeurs manquants pour la colonne brand

In [27]:
# Imputation des valeurs manquantes par "Unknown" pour la colonne brand
# Cela permet de conserver les lignes concernées dans les analyses
# tout en gardant une catégorie identifiable
df_clean['brand'] = df_clean['brand'].fillna('Unknown')

# Vérification : plus aucune valeur manquante dans brand
print(f"Valeurs manquantes dans brand : {df_clean['brand'].isna().sum()}")

Valeurs manquantes dans brand : 0


#### 3.2.2 Vérification de la colonne item_status

In [28]:
# Affichage des statuts uniques des lignes produits
df_clean['item_status'].unique()

array(['Shipped', 'Complete', 'Processing', 'Returned', 'Cancelled'],
      dtype=object)

In [29]:
# Pourcentage des statuts
df_clean['item_status'].value_counts(normalize=True) * 100

item_status
Shipped       29.60
Complete      25.31
Processing    18.23
Cancelled     14.29
Returned      12.57
Name: proportion, dtype: float64

<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observation :

- On observe 5 statuts différents. Une part importante des commandes est expédiée ou reçue, mais une part non négligeable est encore en cours ou annulée.

- **Convention du projet** : on utilisera uniquement `Complete` pour les ventes et `Returned` pour les retours dans les calculs de KPI.
</details

#### 3.2.3 Vérification de la colonne shipped_at et de la colonne delivered_at

In [30]:
# Vérifier les statuts associés aux valeurs manquantes de shipped_at
print("Statuts pour les valeurs manquantes de shipped_at")
print(df_clean[df_clean['shipped_at'].isna()]['order_status'].value_counts())

print()

# Vérifier les statuts associés aux valeurs manquantes de delivered_at
print("Statuts pour les valeurs manquantes de delivered_at")
print(df_clean[df_clean['delivered_at'].isna()]['order_status'].value_counts())

Statuts pour les valeurs manquantes de shipped_at
order_status
Processing    306
Cancelled     240
Name: count, dtype: int64

Statuts pour les valeurs manquantes de delivered_at
order_status
Shipped       497
Processing    306
Cancelled     240
Name: count, dtype: int64


<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observation :

- Les valeurs manquantes de `shipped_at` correspondent principalement aux statuts `Processing` et `Cancelled` : cohérent, ces commandes n'ont pas encore été envoyées.

- Les valeurs manquantes de `delivered_at` correspondent aux statuts `Processing`, `Cancelled` et `Shipped` : cohérent, ces commandes ne sont pas encore livrées.

- Ces valeurs manquantes sont structurelles et ne traduisent pas un problème de qualité des données.
</details>

### 3.3 Traitement des doublons

#### 3.3.1 Identification des vrais doublons

In [31]:
''' 
Comme nous l'avons vu dans le cours, la méthode duplicated() permet d'identifier les doublons. 
Nous avons obtenu comme résultat que toutes les lignes ont la valeur False, ce peut signifier qu'il n'y a pas de doublon 
et que les lignes du jeu de donnée fournis ne sont pas dupliqué sur l'ensemble des colonnes.
'''

# Comptage des lignes intégralement dupliquées
nombre_doublons = df_clean.duplicated().sum()
print(f"Il y a {nombre_doublons} ligne(s) intégralement dupliquée(s)")

Il y a 0 ligne(s) intégralement dupliquée(s)


<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observation :

- Aucun doublon intégral détecté. Chaque ligne correspond à une ligne de commande unique.

- Cela renforce la fiabilité des calculs de KPI futurs.
</details>

#### 3.3.2 Traitement des faux doublons

In [32]:
'''
Test sur la colonne order_item_id
'''

# Vérification : order_item_id doit être unique (identifiant de ligne produit)
doublons_item_id = df_clean.duplicated(subset=['order_item_id']).sum()
print(f"Doublons sur order_item_id : {doublons_item_id}")

Doublons sur order_item_id : 0


In [33]:
'''
Test sur les colonnes order_id et product_id
'''

# Vérification : un même produit ne doit pas apparaître deux fois dans la même commande
doublons_order_product = df_clean.duplicated(subset=['order_id', 'product_id']).sum()
print(f"Doublons sur order_id & product_id : {doublons_order_product}")

Doublons sur order_id & product_id : 0


<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observation :

- Aucun doublon sur `order_item_id` : chaque ligne de commande est unique.

- Aucun doublon sur `order_id + product_id` : chaque produit n'apparaît qu'une seule fois par commande.

- L'intégrité des données est confirmée. Les KPI (CA, marge, taux de retour) seront calculés sur des données fiables.
</details>

# 4. Conversion des données

### 4.1 Conversion des dates

In [35]:
'''  
Conversion des colonnes au format datetime
Cela permet à Python de reconnaître les colonnes comme des dates
pour pouvoir faire des analyses temporelles (année, mois ou les délais par exemple)
'''

#  Conversion des colonnes au format datetime
df_clean['item_created_at'] = pd.to_datetime(df_clean['item_created_at'])
df_clean['order_created_at'] = pd.to_datetime(df_clean['order_created_at'])
df_clean['shipped_at'] = pd.to_datetime(df_clean['shipped_at'])
df_clean['delivered_at'] = pd.to_datetime(df_clean['delivered_at'])

# Vérification des types de données
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   order_id          1679 non-null   int64              
 1   order_item_id     1679 non-null   int64              
 2   product_id        1679 non-null   int64              
 3   item_created_at   1679 non-null   datetime64[ns, UTC]
 4   item_status       1679 non-null   object             
 5   sale_price        1679 non-null   float64            
 6   cost              1679 non-null   float64            
 7   category          1679 non-null   object             
 8   department        1679 non-null   object             
 9   brand             1679 non-null   object             
 10  product_name      1679 non-null   object             
 11  order_status      1679 non-null   object             
 12  order_created_at  1679 non-null   datetime64[ns, UTC]
 13  shi

### 4.2 Création des variables temporelles

In [36]:
# Création des colonnes year et month à partir de item_created_at
# Cela simplifiera toutes les analyses temporelles qui suivent
df_clean['year']  = df_clean['item_created_at'].dt.year
df_clean['month'] = df_clean['item_created_at'].dt.month

# Affichage de 5 lignes aléatoires pour vérification
df_clean.sample(5)

,order_id,order_item_id,product_id,item_created_at,item_status,sale_price,cost,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,user_id,gender,country,state,city,year,month
550,46154,67071,15505,2023-10-25 05:31:49+00:00,Returned,26.95,13.96,Plus,Women,SmartWool,Smartwool Women's Basic Thigh High Sock,Returned,2023-10-24 07:32:00+00:00,2023-10-24 22:18:00+00:00,2023-10-28 01:50:00+00:00,37022,F,France,Nouvelle-Aquitaine,Saint-Jean-d'Illac,2023,10
1618,3003,4292,2131,2024-12-11 14:25:23+00:00,Shipped,70.00,35.77,Fashion Hoodies & Sweatshirts,Women,Life Is Good,Life is good. Womens Softwash Zippity - LIG - ...,Shipped,2024-12-11 17:30:00+00:00,2024-12-14 06:43:00+00:00,NaT,2480,F,France,Grand Est,Habsheim,2024,12
99,79754,115801,11976,2023-03-04 03:08:17+00:00,Shipped,19.99,11.51,Intimates,Women,Bslingerie,Bslingerie Womens Satin Boned Bridal Bustier C...,Shipped,2023-03-04 06:48:00+00:00,2023-03-06 01:20:00+00:00,NaT,63786,F,France,Île-de-France,Saint-Martin-la-Garenne,2023,3
1034,26617,38624,14849,2024-06-03 01:50:25+00:00,Processing,26.98,11.22,Maternity,Women,Motherhood Maternity,Motherhood Maternity: Full Coverage All Over L...,Processing,2024-06-01 03:20:00+00:00,NaT,NaT,21410,F,France,Île-de-France,Paris,2024,6
746,57041,82942,8164,2024-01-18 17:21:02+00:00,Processing,24.23,15.12,Suits,Women,Allegra K,Allegra K Women Elastic Waist Pants Sweetheart...,Processing,2024-01-14 18:03:00+00:00,NaT,NaT,45672,F,France,Île-de-France,Chevilly-Larue,2024,1


### 4.3 Vérification de la période temporelle

In [37]:
'''
On s'assure que toutes les dates sont bien dans la période 2023-2024
L'énoncé demande explicitement cette vérification.
'''

# Vérification des bornes temporelles
print(f"Date minimum : {df_clean['item_created_at'].min()}")
print(f"Date maximum : {df_clean['item_created_at'].max()}")
print()
print("Les données doivent couvrir du 01/01/2023 au 31/12/2024")

Date minimum : 2023-01-01 06:18:03+00:00
Date maximum : 2024-12-31 10:40:48+00:00

Les données doivent couvrir du 01/01/2023 au 31/12/2024


<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observation :

- Les données couvrent bien la période demandée : du 01/01/2023 au 31/12/2024.

- Cette étape garantit la cohérence du sous-périmètre d'analyse demandé par la direction e-commerce.
</details>

### 4.4  Définition du périmètre ventes

In [38]:
# Convention du projet : item_status == "Complete" = vente finalisée
# On crée df_sales une seule fois ici pour éviter les redéfinitions
# dans les sections suivantes

# Filtre des ventes complètes
df_sales = df_clean[df_clean['item_status'] == 'Complete'].copy()

# Afficher les dimensions du dataset filtré
print(f"Dimensions du dataset ventes : {df_sales.shape[0]} lignes x {df_sales.shape[1]} colonnes")

# Affichage de 5 lignes aléatoires
df_sales.sample(5)

Dimensions du dataset ventes : 425 lignes x 22 colonnes


,order_id,order_item_id,product_id,item_created_at,item_status,sale_price,cost,category,department,brand,product_name,order_status,order_created_at,shipped_at,delivered_at,user_id,gender,country,state,city,year,month
916,70548,102455,808,2024-04-09 04:33:15+00:00,Complete,99.50,46.47,Sweaters,Women,Calvin Klein,Calvin Klein Women's Plus-Size Women's Graphic...,Complete,2024-04-08 06:43:00+00:00,2024-04-09 06:51:00+00:00,2024-04-10 03:13:00+00:00,56441,F,France,Île-de-France,Argenteuil,2024,4
750,69792,101388,1787,2024-01-20 11:02:43+00:00,Complete,49.50,21.04,Fashion Hoodies & Sweatshirts,Women,Roxy,Roxy Bliss Fleece Hoodie Sweatshirt Turquoise,Complete,2024-01-16 12:06:00+00:00,2024-01-18 05:17:00+00:00,2024-01-19 02:56:00+00:00,55831,F,France,Île-de-France,Villiers-sur-Marne,2024,1
631,45001,65378,10565,2023-12-05 21:45:33+00:00,Complete,56.00,30.46,Intimates,Women,Wacoal,Wacoal Women's Feather Embroidery Full Figure ...,Complete,2023-12-04 00:40:00+00:00,2023-12-05 19:41:00+00:00,2023-12-07 01:01:00+00:00,36125,F,France,Hauts-de-France,Marcq-en-Barœul,2023,12
306,11817,17089,10838,2023-06-18 16:21:43+00:00,Complete,12.99,6.49,Intimates,Women,Fruit of the Loom,Fruit of the Loom Women's 6-Pack Cotton Bikini,Complete,2023-06-16 17:35:00+00:00,2023-06-17 00:54:00+00:00,2023-06-21 10:08:00+00:00,9583,F,France,Hauts-de-France,Cléry-sur-Somme,2023,6
990,20575,29767,7405,2024-05-14 12:03:58+00:00,Complete,89.00,33.29,Skirts,Women,Jones New York,Jones New York Women's Pleated Skirt,Complete,2024-05-14 15:00:00+00:00,2024-05-17 12:13:00+00:00,2024-05-18 15:55:00+00:00,16598,F,France,Nouvelle-Aquitaine,Saint-Junien,2024,5


# 5. Analyse Exploratoire (EDA)

### 5.1 Distribution du prix de vente

#### 5.1.1 Statistiques descriptives

In [39]:
# Statistiques descriptives du prix de vente 
# On utilise df_sales : uniquement les ventes finalisées
df_sales["sale_price"].describe()

count    425.00
mean      55.35
std       71.41
min        1.82
25%       18.97
50%       36.29
75%       62.36
max      903.00
Name: sale_price, dtype: float64

<details>
<summary><font size="3" color="#ff7373"><b>Cliquez pour afficher les constats</b></font></summary>

### Observations :

- Les prix de vente présentent une forte dispersion et une distribution asymétrique à droite, avec quelques produits très coûteux qui influencent la moyenne.

- On observe des produits très accessibles, mais aussi des produits premium avec un prix maximum à 903 €.

-  On peut supposer que le catalogue semble combiner une offre majoritairement accessible avec un nombre limité de produits premium, traduisant un positionnement sur plusieurs segments de marché.
</details>